In [1]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import contextily as ctx

In [8]:
# Generate static image for testing
# --- Configuration ---
base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
out_dir = "/export/lv9/projects/dws/results/animation/benthos"
spinup = "spinup_01"
date_str = "201501"
variable = "Y1c" # Benthic functional group

# Fixed map window: Dutch Wadden Sea to Ems-Dollard
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# --- Load Data ---
file_path = os.path.join(base_dir, spinup, f"dws_500m.3d.{date_str}.nc")
ds = xr.open_dataset(file_path)

# Extract variable. If the variable is 3D (e.g., pelagic), we safely isolate the bottom layer.
data_var = ds[variable]
if 'level' in data_var.dims:
    data_var = data_var.isel(level=-1) 

# Select the first time step in the daily output file
data_slice = data_var.isel(time=31)

# Extract 2D coordinate arrays and mathematically fill any spatial gaps (NaNs) 
# usually found in the masked land boundaries of the domain mesh.
lon_da = ds.lonc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')
lat_da = ds.latc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')

lon = lon_da.values
lat = lat_da.values

# --- Plotting ---
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())
ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

## Add map features
#ax.add_feature(cfeature.LAND, zorder=2, edgecolor='k', facecolor='lightgray')
#ax.add_feature(cfeature.COASTLINE, zorder=2)

# Plot spatial distribution
mesh = ax.pcolormesh(lon, lat, data_slice.values,
                     transform=ccrs.PlateCarree(),
                     cmap=cmocean.cm.dense,
                     shading='nearest',
                     zorder=10)

# Add the high-resolution aerial background using Esri Imagery
ctx.add_basemap(ax, 
                crs=ax.projection.to_string(), 
                source=ctx.providers.Esri.WorldImagery, 
                zorder=1,
               )

txt = ax.texts[-1]
txt.set_position([0.99,0.02])
txt.set_ha('right')

# Formatting
cbar = plt.colorbar(mesh, ax=ax, shrink=0.7, pad=0.05)

# Dynamically fetch units if available in the netCDF metadata
units = ds[variable].attrs.get('units', '')
cbar.set_label(f"{variable} [{units}]")

# Extract human-readable time
time_str = str(data_slice.time.values).split('T')[0]
plt.title(f"Spatial Distribution of {variable} - {spinup} ({time_str})")

# Save output
output_filename = f"{variable}_{spinup}_{date_str}_spatial.png"
plt.savefig(os.path.join(out_dir,output_filename), dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved static plot to {output_filename}")
ds.close()

Saved static plot to Y1c_spinup_01_201501_spatial.png
